# Launch Goblin King Workloads From JupyterHub

This workbook uses the JupyterHub API token from the single-user environment to declare a Python function goblin, validate it, run it, and then access a long-running service through Goblin King.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request

package_spec = importlib.util.find_spec("goblin_king")
if package_spec is None:
    package = os.environ.get(
        "GOBLIN_KING_NOTEBOOK_PACKAGE",
        "git+https://github.com/tashabits/goblin-king.git",
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from goblin_king import GoblinKingNotebookClient  # noqa: E402

GOBLIN_KING_API_URL = os.environ.get(
    "GOBLIN_KING_API_URL",
    "http://goblin-king-api.default.svc.cluster.local:8000",
).rstrip("/")
JUPYTERHUB_TOKEN = os.environ["JUPYTERHUB_API_TOKEN"]
client = GoblinKingNotebookClient(api_url=GOBLIN_KING_API_URL, token=JUPYTERHUB_TOKEN)

def goblin_request(path, method="GET", payload=None):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {"Authorization": f"Bearer {JUPYTERHUB_TOKEN}"}
    if body is not None:
        headers["Content-Type"] = "application/json"
    request = urllib.request.Request(
        GOBLIN_KING_API_URL + path,
        data=body,
        headers=headers,
        method=method,
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            text = response.read().decode("utf-8")
            return json.loads(text) if text else {"status": response.status}
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8")
        raise RuntimeError(f"Goblin King API returned {error.code}: {detail}") from error

GOBLIN_KING_API_URL

In [ ]:
def workbook_hello(payload):
    name = payload.get("name", "Workbook")
    return {
        "message": f"Hello {name}",
        "name_length": len(name),
    }

workbook_goblin = client.declare(
    workbook_hello,
    kind="notebook.workbook-hello",
    display_name="Workbook Hello",
    timeout_seconds=30,
)
workbook_goblin.record

In [ ]:
validation = workbook_goblin.validate({"name": "Validation"})
validation["validation"]

In [ ]:
run = workbook_goblin.run({"name": "JupyterHub"})
run["run"]["result"]

In [ ]:
goblins = goblin_request("/goblins")
[item["kind"] for item in goblins if item["kind"].startswith("notebook.")]

In [ ]:
service = goblin_request(
    "/services/long-running",
    method="POST",
    payload={
        "kind": "example.long-hello",
        "base_url": "http://goblin-king-long-hello",
        "probe_path": "/hello",
        "project_id": "default",
    },
)
service

In [ ]:
probe = goblin_request(
    f"/services/long-running/{service['id']}/probe",
    method="POST",
)
probe["response"]["json"]

In [ ]:
proxied = goblin_request(f"/services/long-running/{service['id']}/proxy/hello")
proxied